# Extraction Results Analysis

This notebook provides functions to analyze and visualize extraction results from the OCR database.
The main function creates a pivot table showing all extracted field values across different runs and model instances.

In [1]:
# Import required libraries
import pandas as pd
import sys
import os

# Add the project root to Python path
sys.path.append('/Users/tobig/code/MasterThesis/OCR/code')

# Import database modules
from db import get_connection, PaperDAO, ModelDAO, ExtractionDAO

In [2]:
def create_extraction_results_table():
    """
    Creates a pivot table with all extraction results.
    
    Returns:
        pandas.DataFrame: Table with arxiv_id, run_id, model_instance_id as first columns,
                         followed by all unique field names as columns containing extracted values
    """
    conn = None
    try:
        conn = get_connection()
        
        # Query to get all extraction results with paper and model information
        query = """
        SELECT 
            p.arxiv_id,
            er.id as run_id,
            ef.model_instance_id,
            m.name as model_name,
            m.provider as model_provider,
            er.run_date,
            ef.field_name,
            ef.value,
            ef.confidence
        FROM extracted_fields ef
        JOIN extraction_runs er ON ef.run_id = er.id
        JOIN papers p ON er.paper_id = p.id
        JOIN models m ON er.model_id = m.id
        ORDER BY p.arxiv_id, er.id, ef.model_instance_id, ef.field_name
        """
        
        # Execute query and create DataFrame
        df = pd.read_sql_query(query, conn)
        
        if df.empty:
            print("No extraction results found in database")
            return pd.DataFrame()
        
        print(f"Query returned {len(df)} rows")
        print(f"Available columns: {list(df.columns)}")
        
        # Create a unique identifier for each row (combination of run_id and model_instance_id)
        df['row_id'] = df['run_id'].astype(str) + '_' + df['model_instance_id'].astype(str)
        
        # Define metadata columns
        metadata_cols = ['arxiv_id', 'run_id', 'model_instance_id', 'model_name', 'model_provider', 'run_date']
        
        print(f"Using metadata columns: {metadata_cols}")
        
        # Get unique combinations of our identifier columns
        metadata_df = df[metadata_cols + ['row_id']].drop_duplicates()
        
        # Create pivot table with field values
        pivot_df = df.pivot_table(
            index='row_id',
            columns='field_name',
            values='value',
            aggfunc='first'  # In case of duplicates, take first value
        ).reset_index()
        
        print(f"Pivot table columns: {list(pivot_df.columns)}")
        print(f"Metadata table columns: {list(metadata_df.columns)}")
        
        # Merge metadata back with pivot table
        result_df = metadata_df.merge(pivot_df, on='row_id', how='left')
        
        print(f"Result table columns after merge: {list(result_df.columns)}")
        
        # Drop the temporary row_id column
        result_df = result_df.drop('row_id', axis=1)
        
        # Reorder columns: metadata first, then field columns
        # Only include metadata columns that actually exist in the result
        available_metadata_cols = [col for col in metadata_cols if col in result_df.columns]
        field_columns = [col for col in result_df.columns if col not in available_metadata_cols]
        final_columns = available_metadata_cols + sorted(field_columns)
        
        print(f"Final column order: {final_columns[:10]}...")  # Show first 10
        
        result_df = result_df[final_columns]
        
        return result_df
        
    except Exception as e:
        print(f"Error creating extraction results table: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

In [3]:
def create_simplified_extraction_table(min_run_id=0):
    """
    Creates a simplified pivot table with just arxiv_id, run_id, model_instance_id and field values.
    
    Returns:
        pandas.DataFrame: Simplified table with arxiv_id as first column
    """
    conn = None
    try:
        conn = get_connection()
        
        # Simplified query
        query = f"""
        SELECT
            p.arxiv_id,
            er.id as run_id,
            ef.model_instance_id,
            ef.field_name,
            ef.value
        FROM extracted_fields ef
        JOIN extraction_runs er ON ef.run_id = er.id
        JOIN papers p ON er.paper_id = p.id
        WHERE er.id >= {min_run_id}
        ORDER BY er.id, p.arxiv_id, ef.model_instance_id, ef.field_name
        """
        
        df = pd.read_sql_query(query, conn)
        
        if df.empty:
            print("No extraction results found in database")
            return pd.DataFrame()
        
        print(f"Query returned {len(df)} rows")
        print(f"Available columns: {list(df.columns)}")
        
        # Create unique row identifier
        df['row_key'] = df['arxiv_id'] + '_' + df['run_id'].astype(str) + '_' + df['model_instance_id'].astype(str)
        
        # Keep metadata
        metadata = df[['arxiv_id', 'run_id', 'model_instance_id', 'row_key']].drop_duplicates()
        
        # Pivot the field values
        pivot = df.pivot_table(
            index='row_key',
            columns='field_name',
            values='value',
            aggfunc='first'
        ).reset_index()
        
        # Merge and clean up
        result = metadata.merge(pivot, on='row_key')
        result = result.drop('row_key', axis=1)
        
        # Ensure arxiv_id is first column
        cols = result.columns.tolist()
        cols = ['arxiv_id', 'run_id', 'model_instance_id'] + [c for c in cols if c not in ['arxiv_id', 'run_id', 'model_instance_id']]
        result = result[cols]
        
        return result
        
    except Exception as e:
        print(f"Error creating simplified extraction table: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

In [4]:
# Test the function
print("Creating extraction results table...")
results_table = create_extraction_results_table()

if not results_table.empty:
    print(f"\nTable created successfully!")
    print(f"Shape: {results_table.shape}")
    print(f"\nColumns: {list(results_table.columns)}")
    print(f"\nFirst few rows:")
    print(results_table.head())
else:
    print("No data found or error occurred")

Creating extraction results table...


/var/folders/ry/w884b67x1r316pnknj1yxy100000gn/T/ipykernel_15147/25760112.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Query returned 619298 rows
Available columns: ['arxiv_id', 'run_id', 'model_instance_id', 'model_name', 'model_provider', 'run_date', 'field_name', 'value', 'confidence']
Using metadata columns: ['arxiv_id', 'run_id', 'model_instance_id', 'model_name', 'model_provider', 'run_date']
Pivot table columns: ['row_id', 'abstract', 'activated_parameters', 'architecture', 'attention', 'attention_dropout', 'attention_heads', 'authors', 'backbone', 'bandwidth', 'base_model', 'batch_size', 'code_dim', 'codebook', 'confidence', 'contrastive_layers', 'control_type', 'domain', 'dropout', 'environment_types', 'epochs', 'ffn_inner_hidden_size', 'hardware', 'hardware_quantity', 'hardware_utilization', 'hidden_size', 'history_length', 'input_modality', 'input_patch', 'layers', 'learning_rate', 'license', 'masked_layers', 'max_learning_rate', 'min_learning_rate', 'model_name', 'model_size_mb', 'number_of_layers', 'optimizer', 'organization', 'output_modality', 'parameters', 'performance', 'policy_represe

In [5]:
# Test the simplified version
print("Creating simplified extraction results table...")
simple_table = create_simplified_extraction_table(10455)

if not simple_table.empty:
    print(f"\nSimplified table created successfully!")
    print(f"Shape: {simple_table.shape}")
    print(f"\nColumns: {list(simple_table.columns)}")
    print(f"\nFirst few rows:")
    display(simple_table.head())
else:
    print("No data found or error occurred")

Creating simplified extraction results table...
Query returned 20778 rows
Available columns: ['arxiv_id', 'run_id', 'model_instance_id', 'field_name', 'value']

Simplified table created successfully!
Shape: (1225, 25)

Columns: ['arxiv_id', 'run_id', 'model_instance_id', 'architecture', 'authors', 'base_model', 'batch_size', 'confidence', 'domain', 'epochs', 'hardware_quantity', 'hardware_utilization', 'input_modality', 'model_name', 'organization', 'output_modality', 'parameters', 'publication_date', 'references', 'task', 'training_dataset', 'training_dataset_size', 'training_hardware', 'training_time', 'value']

First few rows:


/var/folders/ry/w884b67x1r316pnknj1yxy100000gn/T/ipykernel_15147/167557947.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,arxiv_id,run_id,model_instance_id,architecture,authors,base_model,batch_size,confidence,domain,epochs,...,output_modality,parameters,publication_date,references,task,training_dataset,training_dataset_size,training_hardware,training_time,value
0,2205.15868,10455,0,Transformer (Decoder-only),"{""Wenyi Hong"",""Ming Ding"",""Wendi Zheng"",""Xingh...",CogView2,416,NaN,{Video},n/a,...,{video},9400000000,n/a,NaN,"{""text-to-video generation"",""video generation""}","{""5.4 million captioned videos""}",5400000,n/a,n/a,NaN
1,2112.15283,10456,0,Multimodal Transformer,"{""Han Zhang"",""Weichong Yin"",""Yewei Fang"",""Lanx...",n/a,n/a,NaN,{Multimodal},n/a,...,"{text,image}",10000000000,n/a,NaN,"{""text-to-image synthesis"",""image captioning"",...","{""Chinese Webpages"",""Image Search Engine"",CC,C...",145M image-text pairs,GPU,n/a,NaN
2,2311.07362,10457,0,Multimodal Transformer,"{""Seongyun Lee"",""Sue Hyun Park"",""Yongrae Jo"",""...",LLaVA-1.5 7B,NaN,NaN,{Multimodal},NaN,...,{text},7000000000,NaN,NaN,"{""multimodal hallucination mitigation"",""visual...",NaN,NaN,NaN,NaN,NaN
3,2311.07362,10457,1,Multimodal Transformer,"{""Seongyun Lee"",""Sue Hyun Park"",""Yongrae Jo"",""...",LLaVA-1.5 13B,NaN,NaN,{Multimodal},NaN,...,{text},13000000000,NaN,NaN,"{""multimodal hallucination mitigation"",""visual...",NaN,NaN,NaN,NaN,NaN
4,1911.03864,10458,0,Transformer (Decoder-only),"{""Ofir Press"",""Noah A. Smith"",""Omer Levy""}",Baevski and Auli (2019) transformer,n/a,NaN,{Language},n/a,...,{text},n/a,n/a,NaN,"{""language modeling/generation""}",{WikiText-103},n/a,n/a,n/a,NaN


In [6]:
#format parameters column from string to numeric if the value is numeric - saved old values in a new column "column_name_string"
def format_parameters_column(df, column_name):
    """
    Convert values in the specified column to numeric, ignoring non-numeric values.
    
    Args:
        df (pandas.DataFrame): DataFrame containing the column
        column_name (str): Name of the column to format
        
    Returns:
        pandas.DataFrame: DataFrame with formatted column
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found in DataFrame")
        return df
    # Save original values in a new column
    df[column_name + '_string'] = df[column_name]
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    return df

simple_table = format_parameters_column(simple_table, 'parameters')
simple_table.head(50)


,arxiv_id,run_id,model_instance_id,architecture,authors,base_model,batch_size,confidence,domain,epochs,...,parameters,publication_date,references,task,training_dataset,training_dataset_size,training_hardware,training_time,value,parameters_string
0,2205.15868,10455,0,Transformer (Decoder-only),"{""Wenyi Hong"",""Ming Ding"",""Wendi Zheng"",""Xingh...",CogView2,416,NaN,{Video},n/a,...,9.400000e+09,n/a,NaN,"{""text-to-video generation"",""video generation""}","{""5.4 million captioned videos""}",5400000,n/a,n/a,NaN,9400000000
1,2112.15283,10456,0,Multimodal Transformer,"{""Han Zhang"",""Weichong Yin"",""Yewei Fang"",""Lanx...",n/a,n/a,NaN,{Multimodal},n/a,...,1.000000e+10,n/a,NaN,"{""text-to-image synthesis"",""image captioning"",...","{""Chinese Webpages"",""Image Search Engine"",CC,C...",145M image-text pairs,GPU,n/a,NaN,10000000000
2,2311.07362,10457,0,Multimodal Transformer,"{""Seongyun Lee"",""Sue Hyun Park"",""Yongrae Jo"",""...",LLaVA-1.5 7B,NaN,NaN,{Multimodal},NaN,...,7.000000e+09,NaN,NaN,"{""multimodal hallucination mitigation"",""visual...",NaN,NaN,NaN,NaN,NaN,7000000000
3,2311.07362,10457,1,Multimodal Transformer,"{""Seongyun Lee"",""Sue Hyun Park"",""Yongrae Jo"",""...",LLaVA-1.5 13B,NaN,NaN,{Multimodal},NaN,...,1.300000e+10,NaN,NaN,"{""multimodal hallucination mitigation"",""visual...",NaN,NaN,NaN,NaN,NaN,13000000000
4,1911.03864,10458,0,Transformer (Decoder-only),"{""Ofir Press"",""Noah A. Smith"",""Omer Levy""}",Baevski and Auli (2019) transformer,n/a,NaN,{Language},n/a,...,NaN,n/a,NaN,"{""language modeling/generation""}",{WikiText-103},n/a,n/a,n/a,NaN,n/a
5,1911.03864,10458,1,Transformer (Decoder-only),{n/a},n/a,n/a,NaN,{Language},n/a,...,NaN,n/a,NaN,"{""language modeling/generation""}",{WikiText-103},n/a,n/a,n/a,NaN,n/a
6,1907.02544,10459,0,Generative Adversarial Network (GAN),"{""Jeff Donahue"",""Karen Simonyan""}",BigGAN,n/a,NaN,"{""Image generation"",Vision}",n/a,...,NaN,n/a,NaN,"{""unsupervised representation learning"",""image...",{ImageNet},n/a,n/a,n/a,NaN,n/a
7,2005.08100v1,10460,0,Hybrid CNN-Transformer,"{""Anmol Gulati"",""James Qin"",""Chung-Cheng Chiu""...",n/a,n/a,NaN,{Speech},n/a,...,1.030000e+07,n/a,NaN,"{""speech recognition""}",{LibriSpeech},n/a,n/a,n/a,NaN,10300000
8,2005.08100v1,10460,1,Hybrid CNN-Transformer,"{""Anmol Gulati"",""James Qin"",""Chung-Cheng Chiu""...",n/a,n/a,NaN,{Speech},n/a,...,3.070000e+07,n/a,NaN,"{""speech recognition""}",{LibriSpeech},n/a,n/a,n/a,NaN,30700000
9,2005.08100v1,10460,2,Hybrid CNN-Transformer,"{""Anmol Gulati"",""James Qin"",""Chung-Cheng Chiu""...",n/a,n/a,NaN,{Speech},n/a,...,1.188000e+08,n/a,NaN,"{""speech recognition""}",{LibriSpeech},n/a,n/a,n/a,NaN,118800000


In [7]:
# Example: Export results to CSV
# export_extraction_results('my_extraction_results.csv')

## Expected Field Names

Based on the project structure, the following field names are expected to appear as columns:

```
Model, Organization, Publication date, Domain, Parameters, Parameters notes,
Training compute (FLOP), Training compute notes, Training dataset, 
Training dataset size (datapoints), Dataset size notes, Confidence, Link,
Reference, Citations, Authors, Abstract, Organization categorization,
Country (of organization), Notability criteria, Notability criteria notes,
Epochs, Training time (hours), Training time notes, Training hardware,
Hardware quantity, Hardware utilization, Training compute cost (2023 USD),
Compute cost notes, Training power draw (W), Base model, 
Finetune compute (FLOP), Finetune compute notes, Batch size, 
Batch size notes, Model accessibility, Training code accessibility,
Inference code accessibility, Accessibility notes, Frontier model
```

In [8]:
# Debug: Check what's actually in the database
def check_database_contents():
    """Debug function to check database contents"""
    conn = None
    try:
        conn = get_connection()
        
        # Check papers
        papers_query = "SELECT COUNT(*) as count FROM papers"
        papers_df = pd.read_sql_query(papers_query, conn)
        print(f"Papers in database: {papers_df['count'].iloc[0]}")
        
        # Check models
        models_query = "SELECT COUNT(*) as count FROM models"
        models_df = pd.read_sql_query(models_query, conn)
        print(f"Models in database: {models_df['count'].iloc[0]}")
        
        # Check extraction runs
        runs_query = "SELECT COUNT(*) as count FROM extraction_runs"
        runs_df = pd.read_sql_query(runs_query, conn)
        print(f"Extraction runs in database: {runs_df['count'].iloc[0]}")
        
        # Check extracted fields
        fields_query = "SELECT COUNT(*) as count FROM extracted_fields"
        fields_df = pd.read_sql_query(fields_query, conn)
        print(f"Extracted fields in database: {fields_df['count'].iloc[0]}")
        
        # Show sample data if available
        if fields_df['count'].iloc[0] > 0:
            sample_query = """
            SELECT p.arxiv_id, er.id as run_id, ef.model_instance_id, ef.field_name, ef.value
            FROM extracted_fields ef
            JOIN extraction_runs er ON ef.run_id = er.id
            JOIN papers p ON er.paper_id = p.id
            LIMIT 5
            """
            sample_df = pd.read_sql_query(sample_query, conn)
            print("\nSample data:")
            print(sample_df)
        
    except Exception as e:
        print(f"Error checking database: {e}")
    finally:
        if conn:
            conn.close()

check_database_contents()

Papers in database: 707
Models in database: 3
Extraction runs in database: 10942
Extracted fields in database: 619298

Sample data:
     arxiv_id  run_id  model_instance_id        field_name  \
0  2501.12599       2                  0        model_name   
1  2501.12599       2                  0            domain   
2  2501.12599       2                  0      organization   
3  2501.12599       2                  0           authors   
4  2501.12599       2                  0  publication_date   

                                               value  
0                                 Kimi k1.5 long-CoT  
1  {Language,Multimodal,Vision,Mathematics,Other,...  
2                                          Kimi Team  
3  {"Angang Du","Bofei Gao","Bowei Xing","Changji...  
4                                                n/a  


/var/folders/ry/w884b67x1r316pnknj1yxy100000gn/T/ipykernel_15147/1433732083.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  papers_df = pd.read_sql_query(papers_query, conn)
/var/folders/ry/w884b67x1r316pnknj1yxy100000gn/T/ipykernel_15147/1433732083.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  models_df = pd.read_sql_query(models_query, conn)
/var/folders/ry/w884b67x1r316pnknj1yxy100000gn/T/ipykernel_15147/1433732083.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs_df = pd.read_sql_query(runs_query, co

In [9]:
# Read the Epoch Database - Notable Models CSV file
import pandas as pd

# Load the CSV file
epoch_csv_path = '/Users/tobig/code/MasterThesis/OCR/code/Epoch Database - Notable Models.csv'

try:
    epoch_df = pd.read_csv(epoch_csv_path)
    print(f"Successfully loaded Epoch Database CSV")
    print(f"Shape: {epoch_df.shape}")
    print(f"\nColumns: {list(epoch_df.columns)}")
    print(f"\nFirst few rows:")
    display(epoch_df.head())
    
    print(f"\nData types:")
    print(epoch_df.dtypes)
    
    print(f"\nSample model names:")
    print(epoch_df['Model'].head(10).tolist())
    
except FileNotFoundError:
    print(f"Error: Could not find the CSV file at {epoch_csv_path}")
except Exception as e:
    print(f"Error reading CSV: {e}")

Error: Could not find the CSV file at /Users/tobig/code/MasterThesis/OCR/code/Epoch Database - Notable Models.csv


In [10]:
# Filter the Epoch Database to only include models with arxiv links
import re

def filter_arxiv_models(df):
    """
    Filter DataFrame to only include models with arxiv links.
    
    Args:
        df (pd.DataFrame): Input DataFrame with 'Link' column
        
    Returns:
        pd.DataFrame: Filtered DataFrame with only arxiv models
    """
    if 'Link' not in df.columns:
        print("Error: 'Link' column not found in DataFrame")
        return pd.DataFrame()
    
    # Define arxiv URL pattern
    arxiv_pattern = r'https://arxiv\.org/abs/\d{4}\.\d{5}'
    
    # Filter rows where Link contains arxiv URL
    arxiv_mask = df['Link'].fillna('').str.contains(arxiv_pattern, regex=True, na=False)
    
    filtered_df = df[arxiv_mask].copy()
    
    print(f"Original dataset: {len(df)} models")
    print(f"Models with arxiv links: {len(filtered_df)} models")
    print(f"Removed: {len(df) - len(filtered_df)} models")
    
    return filtered_df

# Apply the filter
if 'epoch_df' in globals():
    arxiv_epoch_df = filter_arxiv_models(epoch_df)
    
    if not arxiv_epoch_df.empty:
        print(f"\nSample arxiv links:")
        sample_links = arxiv_epoch_df['Link'].head(5).tolist()
        for i, link in enumerate(sample_links, 1):
            print(f"{i}. {link}")
        
        print(f"\nFiltered dataset shape: {arxiv_epoch_df.shape}")
        print(f"\nModel names with arxiv links:")
        print(arxiv_epoch_df['Model'].head(10).tolist())
        
        # Display first few rows
        print(f"\nFirst few rows of filtered data:")
        display(arxiv_epoch_df[['Model', 'Organization', 'Publication date', 'Link']].head())
    else:
        print("No models with arxiv links found!")
else:
    print("Please run the CSV loading cell first to load epoch_df")

Please run the CSV loading cell first to load epoch_df


In [11]:
# Extract arXiv ID from the Link column and add as new column
import re

def extract_arxiv_id(df):
    """
    Extract arXiv ID from Link column and add as new column 'arxiv_id'.
    
    Args:
        df (pd.DataFrame): DataFrame with 'Link' column containing arXiv URLs
        
    Returns:
        pd.DataFrame: DataFrame with new 'arxiv_id' column
    """
    if 'Link' not in df.columns:
        print("Error: 'Link' column not found in DataFrame")
        return df
    
    # Pattern to extract arXiv ID from URL
    # Matches patterns like: https://arxiv.org/abs/2504.07866
    arxiv_pattern = r'https://arxiv\.org/abs/(\d{4}\.\d{5}(?:v\d+)?)'
    
    # Extract arXiv IDs
    df = df.copy()
    df['arxiv_id'] = df['Link'].str.extract(arxiv_pattern, expand=False)
    
    # Count successful extractions
    extracted_count = df['arxiv_id'].notna().sum()
    total_count = len(df)
    
    print(f"Successfully extracted arXiv IDs: {extracted_count}/{total_count}")
    
    # Show any failed extractions
    failed_extractions = df[df['arxiv_id'].isna() & df['Link'].notna()]
    if len(failed_extractions) > 0:
        print(f"\nFailed to extract arXiv ID from {len(failed_extractions)} links:")
        for idx, link in failed_extractions['Link'].head(5).items():
            print(f"  {link}")
    
    return df

# Apply arXiv ID extraction to the filtered dataset
if 'arxiv_epoch_df' in globals() and not arxiv_epoch_df.empty:
    arxiv_epoch_df_with_id = extract_arxiv_id(arxiv_epoch_df)
    
    # Show sample results
    print(f"\nDataFrame shape after adding arXiv IDs: {arxiv_epoch_df_with_id.shape}")
    print(f"\nSample arXiv IDs extracted:")
    sample_ids = arxiv_epoch_df_with_id[['Model', 'Link', 'arxiv_id']].head(10)
    display(sample_ids)
    
    # Show the new column position
    print(f"\nColumn 'arxiv_id' added at position: {list(arxiv_epoch_df_with_id.columns).index('arxiv_id')}")
    print(f"Total columns: {len(arxiv_epoch_df_with_id.columns)}")
    
else:
    print("Please run the arXiv filtering cell first to create arxiv_epoch_df")

Please run the arXiv filtering cell first to create arxiv_epoch_df


In [12]:
def get_column_mapping():
    """
    Returns a mapping dictionary from Epoch CSV column names to database field names.
    
    Returns:
        dict: Mapping from Epoch CSV columns to database field names
    """
    return {
        # Keep these as-is
        'Model': 'model_name',
        'arxiv_id': 'arxiv_id',
        
        # Map Epoch CSV columns to database field names
        'Organization': 'organization',
        'Publication date': 'publication_date',
        'Domain': 'domain',
        'Parameters': 'parameters',
        'Parameters notes': 'parameters_notes',
        'Training compute (FLOP)': 'training_compute_flop',
        'Training compute notes': 'training_compute_notes',
        'Training dataset': 'training_dataset',
        'Training dataset size (datapoints)': 'training_dataset_size',
        'Dataset size notes': 'dataset_size_notes',
        'Confidence': 'confidence',
        'Link': 'link',
        'Reference': 'reference',
        'Citations': 'citations',
        'Authors': 'authors',
        'Abstract': 'abstract',
        'Organization categorization': 'organization_categorization',
        'Country (of organization)': 'country_of_organization',
        'Notability criteria': 'notability_criteria',
        'Notability criteria notes': 'notability_criteria_notes',
        'Epochs': 'epochs',
        'Training time (hours)': 'training_time',
        'Training time notes': 'training_time_notes',
        'Training hardware': 'training_hardware',
        'Hardware quantity': 'hardware_quantity',
        'Hardware utilization': 'hardware_utilization',
        'Training compute cost (2023 USD)': 'training_compute_cost_2023_usd',
        'Compute cost notes': 'compute_cost_notes',
        'Training power draw (W)': 'training_power_draw_w',
        'Base model': 'base_model',
        'Finetune compute (FLOP)': 'finetune_compute_flop',
        'Finetune compute notes': 'finetune_compute_notes',
        'Batch size': 'batch_size',
        'Batch size notes': 'batch_size_notes',
        'Model accessibility': 'model_accessibility',
        'Training code accessibility': 'training_code_accessibility',
        'Inference code accessibility': 'inference_code_accessibility',
        'Accessibility notes': 'accessibility_notes',
        'Frontier model': 'frontier_model'
    }

In [13]:
# Optional: Reorder columns to put arxiv_id near the front
def reorder_columns_with_arxiv_id(df, rename_columns=True):
    """
    Reorder columns to put arxiv_id as the second column (after Model) and optionally
    rename columns to match database field names and format parameters.
    
    Args:
        df (pd.DataFrame): DataFrame with arxiv_id column
        rename_columns (bool): Whether to rename columns to match database field names
        format_parameters (bool): Whether to format parameters column for readability
        
    Returns:
        pd.DataFrame: DataFrame with reordered and optionally renamed columns
    """
    if 'arxiv_id' not in df.columns:
        print("Error: 'arxiv_id' column not found")
        return df
    
    df = df.copy()
      
    # Rename columns if requested
    if rename_columns:
        column_mapping = get_column_mapping()
        # Only rename columns that exist in the DataFrame
        existing_mapping = {k: v for k, v in column_mapping.items() if k in df.columns}
        df = df.rename(columns=existing_mapping)
        print(f"Renamed {len(existing_mapping)} columns to match database field names")
    
    # Get current column order
    cols = df.columns.tolist()
    
    # Remove arxiv_id from its current position
    cols.remove('arxiv_id')
    
    # Determine the model column name (could be 'Model' or 'model_name')
    model_col = 'model_name' if 'model_name' in cols else 'Model'
    
    # Insert arxiv_id as second column (after model column)
    if model_col in cols:
        model_idx = cols.index(model_col)
        cols.insert(model_idx + 1, 'arxiv_id')
    else:
        # If no model column, put at the beginning
        cols.insert(0, 'arxiv_id')
    
    return df[cols]

# Apply column reordering and renaming
if 'arxiv_epoch_df_with_id' in globals():
    arxiv_epoch_df_final = reorder_columns_with_arxiv_id(arxiv_epoch_df_with_id, rename_columns=True)
    
    print(f"Reordered and renamed columns. First 5 columns: {list(arxiv_epoch_df_final.columns[:5])}")
    
    # Display sample with new column order and names
    print(f"\nSample data with reordered and renamed columns:")
    display(arxiv_epoch_df_final[['model_name', 'arxiv_id', 'organization', 'publication_date', 'link']].head())
    
    # Show a few more renamed columns for verification
    print(f"\nOther renamed columns (sample):")
    renamed_cols = ['parameters', 'training_dataset', 'training_hardware', 'base_model']
    available_renamed_cols = [col for col in renamed_cols if col in arxiv_epoch_df_final.columns]
    if available_renamed_cols:
        print(f"Available renamed columns: {available_renamed_cols}")
        display(arxiv_epoch_df_final[['model_name', 'arxiv_id'] + available_renamed_cols].head())
    
else:
    print("Please run the previous cell first to extract arXiv IDs")

Please run the previous cell first to extract arXiv IDs


In [14]:
# Optional: Compare column names between database and renamed epoch data
def compare_column_names():
    """Compare column names between database extraction results and renamed Epoch data."""
    if 'simple_table' in globals() and 'arxiv_epoch_df_final' in globals():
        db_cols = set(simple_table.columns)
        epoch_cols = set(arxiv_epoch_df_final.columns)
        
        print("Column name comparison:")
        print(f"Database columns: {len(db_cols)}")
        print(f"Epoch columns: {len(epoch_cols)}")
        
        common_cols = db_cols.intersection(epoch_cols)
        db_only = db_cols - epoch_cols
        epoch_only = epoch_cols - db_cols
        
        print(f"\nCommon columns ({len(common_cols)}): {sorted(common_cols)}")
        print(f"\nDatabase-only columns ({len(db_only)}): {sorted(db_only)}")
        print(f"\nEpoch-only columns ({len(epoch_only)}): {sorted(epoch_only)}")
    else:
        print("Please ensure both 'simple_table' and 'arxiv_epoch_df_final' are available")

# Run the comparison
compare_column_names()

Please ensure both 'simple_table' and 'arxiv_epoch_df_final' are available


In [15]:
# Perform inner join on arxiv_id and reorder columns for comparison
def create_comparison_table():
    """
    Performs an inner join between database extraction results and Epoch database on arxiv_id.
    Reorders columns to group extracted and epoch fields side by side for easy manual comparison.
    
    Returns:
        pandas.DataFrame: Joined table with columns grouped for comparison
    """
    # Get the data
    if 'simple_table' not in globals() or 'arxiv_epoch_df_final' not in globals():
        print("Error: Please run the previous cells to create simple_table and arxiv_epoch_df_final")
        return pd.DataFrame()
    
    # Perform inner join on arxiv_id
    comparison_df = simple_table.merge(
        arxiv_epoch_df_final, 
        on='arxiv_id', 
        how='inner',
        suffixes=('_extracted', '_epoch')
    )
    
    print(f"Found {len(comparison_df)} papers present in both databases")
    print(f"Unique arxiv_ids in comparison: {comparison_df['arxiv_id'].nunique()}")
    
    # Define field pairs for side-by-side comparison
    # Format: (extracted_field, epoch_field, display_name)
    field_pairs = [
        ('model_name', 'model_name', 'Model Name'),
        ('organization', 'organization', 'Organization'),
        ('parameters', 'parameters', 'Parameters'),
        ('authors', 'authors', 'Authors'),
        ('training_dataset', 'training_dataset', 'Training Dataset'),
        ('training_dataset_size', 'training_dataset_size', 'Training Dataset Size'),
        ('training_hardware', 'training_hardware', 'Training Hardware'),
        ('hardware_quantity', 'hardware_quantity', 'Hardware Quantity'),
        ('training_time', 'training_time', 'Training Time'),
        ('base_model', 'base_model', 'Base Model'),
        ('domain', 'domain', 'Domain'),
        ('publication_date', 'publication_date', 'Publication Date'),
    ]
    
    # Start with identifier columns
    ordered_columns = ['arxiv_id', 'run_id', 'model_instance_id']
    
    # Add field pairs side by side
    for extracted_field, epoch_field, display_name in field_pairs:
        # Add extracted field
        extracted_col = f"{extracted_field}_extracted" if f"{extracted_field}_extracted" in comparison_df.columns else extracted_field
        if extracted_col in comparison_df.columns:
            ordered_columns.append(extracted_col)
        
        # Add epoch field right after extracted field
        epoch_col = f"{epoch_field}_epoch" if f"{epoch_field}_epoch" in comparison_df.columns else epoch_field
        if epoch_col in comparison_df.columns and epoch_col != extracted_col:
            ordered_columns.append(epoch_col)
    
    # Add any remaining extracted fields
    remaining_extracted = [col for col in comparison_df.columns if col.endswith('_extracted') and col not in ordered_columns]
    ordered_columns.extend(remaining_extracted)
    
    # Add any remaining epoch fields
    remaining_epoch = [col for col in comparison_df.columns if col.endswith('_epoch') and col not in ordered_columns]
    ordered_columns.extend(remaining_epoch)
    
    # Add any other remaining columns
    remaining_other = [col for col in comparison_df.columns if col not in ordered_columns]
    ordered_columns.extend(remaining_other)
    
    # Reorder the DataFrame
    comparison_df_ordered = comparison_df[ordered_columns]
    
    return comparison_df_ordered

# Create the comparison table
comparison_table = create_comparison_table()

if not comparison_table.empty:
    print(f"\nComparison table created successfully!")
    print(f"Shape: {comparison_table.shape}")
    print(f"\nFirst 10 columns: {list(comparison_table.columns[:10])}")
    print(f"\nSample comparison for first paper:")
    
    # Show a sample comparison for the first paper
    if len(comparison_table) > 0:
        sample_row = comparison_table.iloc[0]
        print(f"\nArxiv ID: {sample_row['arxiv_id']}")
        
        # Show key side-by-side comparisons
        comparison_pairs = [
            ('model_name', 'model_name'),
            ('organization', 'organization'),
            ('parameters', 'parameters'),
            ('training_dataset', 'training_dataset'),
        ]
        
        for field_base, _ in comparison_pairs:
            extracted_col = f"{field_base}_extracted" if f"{field_base}_extracted" in sample_row else field_base
            epoch_col = f"{field_base}_epoch" if f"{field_base}_epoch" in sample_row else field_base
            
            if extracted_col in sample_row and epoch_col in sample_row:
                print(f"\n{field_base.replace('_', ' ').title()}:")
                print(f"  Extracted: {sample_row[extracted_col]}")
                print(f"  Epoch:     {sample_row[epoch_col]}")
else:
    print("No matching papers found or error occurred")

Error: Please run the previous cells to create simple_table and arxiv_epoch_df_final
No matching papers found or error occurred


In [16]:
# add columns which calculate the difference between extracted and epoch values for parameters if they are numeric
# the input should be the prefix of the column names, e.g. "parameters" and "training_dataset_size"

import numpy as np


def add_difference_columns(df, prefix):
    """
    Adds difference columns for numeric fields with the specified prefix.
    
    Args:
        df (pd.DataFrame): DataFrame to modify
        prefix (str): Prefix of the columns to calculate differences for
        
    Returns:
        pd.DataFrame: DataFrame with new difference columns added
    """
    if df.empty:
        print("Input DataFrame is empty, skipping difference calculation")
        return df
    
    # Find all columns with the specified prefix
    extracted_col = f"{prefix}_extracted"
    epoch_col = f"{prefix}_epoch"
    
    if extracted_col not in df.columns or epoch_col not in df.columns:
        print(f"Columns '{extracted_col}' or '{epoch_col}' not found in DataFrame")
        return df
    

    # Create new column names for differences
    diff_col = f"{prefix}_difference"
    
    # Calculate differences only for numeric values
    df[diff_col] = pd.to_numeric(df[extracted_col], errors='coerce') - pd.to_numeric(df[epoch_col], errors='coerce')
    
    return df

# add a column which calculates the percentage difference between extracted and epoch values for parameters if they are numeric
def add_percentage_difference_columns(df, prefix):
    """
    Adds percentage difference columns for numeric fields with the specified prefix.
    
    Args:
        df (pd.DataFrame): DataFrame to modify
        prefix (str): Prefix of the columns to calculate percentage differences for
        
    Returns:
        pd.DataFrame: DataFrame with new percentage difference columns added
    """
    if df.empty:
        print("Input DataFrame is empty, skipping percentage difference calculation")
        return df
    
    # Find all columns with the specified prefix
    extracted_col = f"{prefix}_extracted"
    epoch_col = f"{prefix}_epoch"
    
    if extracted_col not in df.columns or epoch_col not in df.columns:
        print(f"Columns '{extracted_col}' or '{epoch_col}' not found in DataFrame")
        return df
    
    # Create new column names for percentage differences
    percent_diff_col = f"{prefix}_percentage_difference"
    
    # Calculate percentage differences only for numeric values
    df[percent_diff_col] = (
        (pd.to_numeric(df[extracted_col], errors='coerce') - pd.to_numeric(df[epoch_col], errors='coerce')) /
        pd.to_numeric(df[epoch_col], errors='coerce').replace(0, pd.NA) * 100
    )
    
    return df

#add a column which sets all but the first digit to 0 and calculates the difference between extracted and epoch values for parameters if they are numeric (extracted:7321, epoch: 7000 -> difference: 0; extracted:73210, epoch: 7000 -> difference: 66000)
def add_first_digit_difference_columns(df, prefix):
    """
    Adds first digit difference columns for numeric fields with the specified prefix.
    
    Args:
        df (pd.DataFrame): DataFrame to modify
        prefix (str): Prefix of the columns to calculate first digit differences for
        
    Returns:
        pd.DataFrame: DataFrame with new first digit difference columns added
    """
    if df.empty:
        print("Input DataFrame is empty, skipping first digit difference calculation")
        return df
    
    # Find all columns with the specified prefix
    extracted_col = f"{prefix}_extracted"
    epoch_col = f"{prefix}_epoch"
    
    if extracted_col not in df.columns or epoch_col not in df.columns:
        print(f"Columns '{extracted_col}' or '{epoch_col}' not found in DataFrame")
        return df
    
    # Create new column names for first digit differences
    first_digit_diff_col = f"{prefix}_first_digit_difference"
    
    # Calculate first digit differences only for numeric values
    def first_digit_difference(extracted, epoch):
        if pd.isna(extracted) or pd.isna(epoch):
            return pd.NA
        extracted_str = str(int(extracted))
        epoch_str = str(int(epoch))
        
        # Get the first digit and set all others to 0
        extracted_first_digit = int(extracted_str[0]) * 10**(len(extracted_str) - 1)
        epoch_first_digit = int(epoch_str[0]) * 10**(len(epoch_str) - 1)
        
        return extracted_first_digit - epoch_first_digit
    
    df[first_digit_diff_col] = df.apply(
        lambda row: first_digit_difference(row[extracted_col], row[epoch_col]), axis=1
    )
    
    return df

#add a column which does not regard the exponent and only compares the factor of the parameters (extracted: 7.321e3, epoch: 7e3 -> difference: 0; extracted: 7.321e4, epoch: 7e3 -> difference: 0.321)
# the numbers are integers or floats and not necessarily in scientific notation
# the column should be usable to compare the factor of the parameters without regarding the exponent / remove all trailing zeros
# e.g. extracted: 7.321e3, epoch: 7e5 -> difference: 0.321
# e.g. extracted: 7.321e4
def add_factor_difference_columns(df, prefix):
    """
    Adds factor difference columns for numeric fields with the specified prefix.
    
    Args:
        df (pd.DataFrame): DataFrame to modify
        prefix (str): Prefix of the columns to calculate factor differences for
        
    Returns:
        pd.DataFrame: DataFrame with new factor difference columns added
    """
    if df.empty:
        print("Input DataFrame is empty, skipping factor difference calculation")
        return df
    
    # Find all columns with the specified prefix
    extracted_col = f"{prefix}_extracted"
    epoch_col = f"{prefix}_epoch"
    
    if extracted_col not in df.columns or epoch_col not in df.columns:
        print(f"Columns '{extracted_col}' or '{epoch_col}' not found in DataFrame")
        return df
    
    # Create new column names for factor differences
    factor_diff_col = f"{prefix}_factor_difference"

    # Calculate factor differences only for numeric values
    def factor_difference(extracted, epoch):
        if pd.isna(extracted) or pd.isna(epoch):
            return pd.NA
        
        # Convert to float and remove trailing zeros
        extracted_float = float(extracted)
        epoch_float = float(epoch)
        
        # Calculate the factor difference
        if epoch_float == 0:
            return pd.NA
        extracted_factor = extracted_float / (10 ** int(np.log10(extracted_float)))
        epoch_factor = epoch_float / (10 ** int(np.log10(epoch_float)))
        return extracted_factor - epoch_factor
    df[factor_diff_col] = df.apply(
        lambda row: factor_difference(row[extracted_col], row[epoch_col]), axis=1
    )
    return df

# the domain column is not numeric, so we cannot apply the same functions as above
# the domain_extracted column holds value like this: "{Language,Games,Other}"
# the domain_epoch column holds value like this: "Language,Games,Other"
def format_domain_column(df, column_name):
    """
    Formats the domain column by converting it to a consistent string format.
    
    Args:
        df (pandas.DataFrame): DataFrame containing the column
        column_name (str): Name of the column to format
        
    Returns:
        pandas.DataFrame: DataFrame with formatted domain column
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found in DataFrame")
        return df
    
    
    #add a new column which holds 

    # Remove curly braces, quotes, etc. and split by comma
    df[column_name] = df[column_name].astype(str).str.replace(r'[\{\}""]', '', regex=True).str.split(',')

    #add "Multimodal" to the column if it is not present and there are more than 1 domain
    df[column_name] = df[column_name].apply(lambda x: x + ['Multimodal'] if isinstance(x, list) and len(x) > 1 and 'Multimodal' not in x else x)

    # Convert lists to sorted strings
    #df[column_name] = df[column_name].apply(lambda x: ','.join(sorted([item.strip() for item in x])) if isinstance(x, list) else x)

    return df

#format the domain columns in the comparison table
comparison_table = format_domain_column(comparison_table, 'domain_extracted')
comparison_table = format_domain_column(comparison_table, 'domain_epoch')

# add a column which shows the words which exist in both domain columns
comparison_table['domain_overlap'] = comparison_table.apply(
    lambda row: set(row['domain_extracted']) & set(row['domain_epoch']), axis=1
)

# add a column which shows the words which exist in the extracted domain but not in the epoch domain
comparison_table['domain_extracted_not_in_epoch'] = comparison_table.apply(
    lambda row: set(row['domain_extracted']) - set(row['domain_epoch']), axis=1
)

# add a column which shows the words which exist in the epoch domain but not in the extracted domain
comparison_table['domain_epoch_not_in_extracted'] = comparison_table.apply(
    lambda row: set(row['domain_epoch']) - set(row['domain_extracted']), axis=1
)

# add a column which calculates the Jaccard similarity between the two domain columns
comparison_table['domain_jaccard_similarity'] = comparison_table.apply(
    lambda row: len(row['domain_overlap']) / (len(row['domain_extracted']) + len(row['domain_epoch']) - len(row['domain_overlap'])) if (len(row['domain_extracted']) + len(row['domain_epoch']) - len(row['domain_overlap'])) > 0 else 0, axis=1
)

comparison_table = add_difference_columns(comparison_table, 'parameters')
comparison_table = add_percentage_difference_columns(comparison_table, 'parameters')
comparison_table = add_first_digit_difference_columns(comparison_table, 'parameters')
comparison_table = add_factor_difference_columns(comparison_table, 'parameters')

Column 'domain_extracted' not found in DataFrame
Column 'domain_epoch' not found in DataFrame


ValueError: Cannot set a DataFrame without columns to the column domain_overlap

In [ ]:
comparison_table.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1207 entries, 0 to 1206
Data columns (total 74 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   arxiv_id                           1207 non-null   object 
 1   run_id                             1207 non-null   int64  
 2   model_instance_id                  1207 non-null   int64  
 3   model_name_extracted               1203 non-null   object 
 4   model_name_epoch                   1207 non-null   object 
 5   organization_extracted             1142 non-null   object 
 6   organization_epoch                 1207 non-null   object 
 7   parameters_extracted               825 non-null    float64
 8   parameters_epoch                   1029 non-null   float64
 9   authors_extracted                  1131 non-null   object 
 10  authors_epoch                      1203 non-null   object 
 11  training_dataset_extracted         1042 non-null   objec

In [ ]:
# Display the full comparison table
if 'comparison_table' in globals() and not comparison_table.empty:
    print("=" * 100)
    print("COMPLETE COMPARISON TABLE")
    print("=" * 100)
    print(f"Table Shape: {comparison_table.shape[0]} rows × {comparison_table.shape[1]} columns")
    print(f"Unique Papers: {comparison_table['arxiv_id'].nunique()}")
    print("="*100)
    
    # Set pandas display options to show all data
    import pandas as pd
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 50)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    
    # Display the full table
    display(comparison_table)
    
    print("\n" + "=" * 100)
    print("COLUMN SUMMARY")
    print("=" * 100)
    
    # Show summary of each column
    for i, col in enumerate(comparison_table.columns):
        non_null_count = comparison_table[col].notna().sum()
        total_count = len(comparison_table)
        print(f"{i+1:2d}. {col:<40} | {non_null_count:3d}/{total_count:3d} non-null values")
    
    # Reset pandas display options
    pd.reset_option('display.max_columns')
    pd.reset_option('display.max_rows')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
else:
    print("Comparison table not available. Please run the previous cells first.")

COMPLETE COMPARISON TABLE
Table Shape: 1207 rows × 74 columns
Unique Papers: 432


,arxiv_id,run_id,model_instance_id,model_name_extracted,model_name_epoch,organization_extracted,organization_epoch,parameters_extracted,parameters_epoch,authors_extracted,authors_epoch,training_dataset_extracted,training_dataset_epoch,training_dataset_size_extracted,training_dataset_size_epoch,training_hardware_extracted,training_hardware_epoch,hardware_quantity_extracted,hardware_quantity_epoch,training_time_extracted,training_time_epoch,base_model_extracted,base_model_epoch,domain_extracted,domain_epoch,publication_date_extracted,publication_date_epoch,batch_size_extracted,confidence_extracted,epochs_extracted,hardware_utilization_extracted,confidence_epoch,epochs_epoch,hardware_utilization_epoch,batch_size_epoch,architecture,input_modality,output_modality,references,task,value,parameters_string,parameters_notes,training_compute_flop,training_compute_notes,dataset_size_notes,link,reference,citations,abstract,organization_categorization,country_of_organization,notability_criteria,notability_criteria_notes,training_time_notes,training_compute_cost_2023_usd,compute_cost_notes,training_power_draw_w,finetune_compute_flop,finetune_compute_notes,batch_size_notes,model_accessibility,training_code_accessibility,inference_code_accessibility,accessibility_notes,frontier_model,domain_overlap,domain_extracted_not_in_epoch,domain_epoch_not_in_extracted,domain_jaccard_similarity,parameters_difference,parameters_percentage_difference,parameters_first_digit_difference,parameters_factor_difference
0,2205.15868,10455,0,CogVideo,CogVideo,"{""Tsinghua University"",BAAI}","Tsinghua University,Beijing Academy of Artific...",9.400000e+09,9.400000e+09,"{""Wenyi Hong"",""Ming Ding"",""Wendi Zheng"",""Xingh...","Wenyi Hong, Ming Ding, Wendi Zheng, Xinghan Li...","{""5.4 million captioned videos""}",Unspecified unreleased,5400000,5.400000e+06,n/a,NaN,n/a,NaN,n/a,NaN,CogView2,CogView2,[Video],"[Multimodal, Video]",n/a,2022-05-29,416,NaN,n/a,n/a,Speculative,NaN,NaN,NaN,Transformer (Decoder-only),{text},{video},NaN,"{""text-to-video generation"",""video generation""}",NaN,9400000000,NaN,NaN,NaN,"""trained on 5.4 million text-video pairs""",https://arxiv.org/abs/2205.15868,CogVideo: Large-scale Pretraining for Text-to-...,356.0,Large-scale pretrained transformers have creat...,"Academia,Academia","China,China",Historical significance,The world's largest and first opensource large...,NaN,NaN,NaN,NaN,30456000000000000000,6ND = 6*9400000000*5400000=3.0456e+17\n\n(numb...,NaN,Open weights (unrestricted),Open source,Open source,https://github.com/THUDM/CogVideo\nApache 2\nt...,NaN,{Video},{},{Multimodal},0.500000,0.000000e+00,0.000000,0,0.0
1,2112.15283,10456,0,ERNIE-ViLG,ERNIE-ViLG,Baidu Inc.,Baidu,1.000000e+10,1.000000e+10,"{""Han Zhang"",""Weichong Yin"",""Yewei Fang"",""Lanx...","Han Zhang, Weichong Yin, Yewei Fang, Lanxin Li...","{""Chinese Webpages"",""Image Search Engine"",CC,C...",NaN,145M image-text pairs,1.450000e+08,GPU,NaN,n/a,NaN,n/a,NaN,n/a,NaN,[Multimodal],"[Multimodal, Image generation, Vision]",n/a,2021-12-31,n/a,NaN,n/a,n/a,NaN,NaN,NaN,NaN,Multimodal Transformer,"{text,image}","{text,image}",NaN,"{""text-to-image synthesis"",""image captioning"",...",NaN,10000000000,"""To explore the landscape of large-scale pre-t...",NaN,NaN,To explore the landscape of large-scale pre-tr...,https://arxiv.org/abs/2112.15283,ERNIE-ViLG: Unified Generative Pre-training fo...,54.0,Conventional methods for the image-text genera...,Industry,China,SOTA improvement,"""we train a 10-billion parameter ERNIE-ViLG mo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{Multimodal},{},"{Image generation, Vision}",0.333333,0.000000e+00,0.000000,0,0.0
2,2311.07362,10457,0,Volcano 7B,Volcano 13B,"KAIST AI, LG AI Research","Korea University,Korea Advanced Institute of S...",7.000000e+09,1.300000e+10,"{""Seongyun Lee"",""Sue Hyun Park"",""Yongrae Jo"",""...","Seongyun Lee, Sue Hyun Park, Yongrae Jo, Minjo...",NaN,"LAION,SBU,ShareGPT4V,Unspecified unreleased",NaN,NaN,NaN,NVIDIA


COLUMN SUMMARY
 1. arxiv_id                                 | 1207/1207 non-null values
 2. run_id                                   | 1207/1207 non-null values
 3. model_instance_id                        | 1207/1207 non-null values
 4. model_name_extracted                     | 1203/1207 non-null values
 5. model_name_epoch                         | 1207/1207 non-null values
 6. organization_extracted                   | 1142/1207 non-null values
 7. organization_epoch                       | 1207/1207 non-null values
 8. parameters_extracted                     | 825/1207 non-null values
 9. parameters_epoch                         | 1029/1207 non-null values
10. authors_extracted                        | 1131/1207 non-null values
11. authors_epoch                            | 1203/1207 non-null values
12. training_dataset_extracted               | 1042/1207 non-null values
13. training_dataset_epoch                   | 885/1207 non-null values
14. training_dataset_size_extracted  

In [ ]:
#create a function which checks for each arxiv_id if one of values in the difference column is 0
#return the number where one value is 0 and the number of arxiv_ids where none of the values is 0, but also present not n/a
#also return the number of arxiv_ids where the difference column is not present or n/a, for those, check if the _extracted or _epoch columns are n/a (i want to have a statistic about which columns caused the n/a how many times, and also when for a arxiv_id the difference column is not present, how many of those have n/a in the _extracted or _epoch columns)
def check_zero_differences(df, prefix='parameters'):
    """
    Checks for each arxiv_id if any of the difference values for the specified prefix is 0.
    
    Args:
        df (pd.DataFrame): DataFrame to check
        prefix (str): Prefix of the difference columns to check
        
    Returns:
        tuple: (count_with_zero, count_without_zero)
    """
    if df.empty:
        print("Input DataFrame is empty, skipping zero difference check")
        return 0, 0
    
    diff_col = f"{prefix}_difference"
    
    if diff_col not in df.columns:
        print(f"Column '{diff_col}' not found in DataFrame")
        return 0, 0
    
    # Group by arxiv_id and check for zero differences
    zero_counts = df.groupby('arxiv_id')[diff_col].apply(lambda x: (x == 0).any())

    count_with_zero = zero_counts.sum()  # Count arxiv_ids with at least one zero difference
    count_without_zero = (zero_counts == False).sum()  # Count arxiv_ids with no zero differences
    count_na_or_missing = df[diff_col].isna().sum()  # Count arxiv_ids with NA in the difference column
    extracted_col = f"{prefix}_extracted"
    epoch_col = f"{prefix}_epoch"
    extracted_string = 'parameters_string'  # Assuming this is the original string column before formatting
    parameters_first_digit_difference = 'parameters_first_digit_difference'
    parameters_factor_difference = 'parameters_factor_difference'

    count_extracted_na = df[extracted_col].isna().sum()  # Count NA in extracted column
    count_epoch_na = df[epoch_col].isna().sum()  # Count NA in epoch column
    count_na_or_missing += count_extracted_na + count_epoch_na  # Total NA across relevant columns
    
    
    # Count combinations of NA values
    count_extracted_na_and_epoch_na = df[(df[extracted_col].isna()) & (df[epoch_col].isna())].shape[0] # arxiv_ids where both extracted and epoch are n/a
    count_extracted_na_and_diff_na = df[(df[extracted_col].isna()) & (df[diff_col].isna())].shape[0] # arxiv_ids where extracted and difference are n/a
    count_epoch_na_and_diff_na = df[(df[epoch_col].isna()) & (df[diff_col].isna())].shape[0] # arxiv_ids where epoch and difference are n/a
    count_extracted_na_and_epoch_na_and_diff_na = df[(df[extracted_col].isna()) & (df[epoch_col].isna()) & (df[diff_col].isna())].shape[0] # arxiv_ids where all three are n/a
    count_extracted_na_only = df[(df[extracted_col].isna()) & (df[epoch_col].notna())].shape[0] # arxiv_ids where extracted is n/a but epoch is not
    count_epoch_na_only = df[(df[epoch_col].isna()) & (df[extracted_col].notna())].shape[0] # arxiv_ids where epoch is n/a but extracted is not


    # compare extract_col to "parameters_string"
    #count how man yarxiv_ids have n/a in extracted_col but not in extracted_string
    # arxiv_ids where extracted is n/a but string is not Error:  'str' object has no attribute 'notna'
    #count_extracted_na_and_not_string_na = df[(df[extracted_col].isna()) & (df[extracted_string].notna())].shape[0]
    #print(f"  - Arxiv IDs with missing in '{extracted_col}' but not in '{extracted_string}': {count_extracted_na_and_not_string_na}")

    # one arxiv_id can be in multiple columns of the dataframe, with different difference values
    unique_arxiv_ids = df['arxiv_id'].unique()  # get unique arxiv_ids
    # Count unique arxiv_ids
    count_unique_arxiv_ids = df['arxiv_id'].nunique()  # number of unique arxiv_ids

    # count for each arxiv_id if any of the difference values is 0
    count_correct = 0
    count_correct_na = 0
    count_incorrect = 0
    count_missing_data_epoch = 0
    count_missing_data_extracted = 0
    count_extracted_na_but_string_not_na = 0
    count_incorrect_digits = 0
    count_incorrect_exponent = 0

    count_correct_domain = 0  # to track arxiv_ids with correct domain values
    count_multimodal_only = 0  # to track arxiv_ids with only "Multimodal" in domain_overlap
    count_additional_domain = 0  # to track arxiv_ids with incorrect domain values
    count_missing_domain_no_matches = 0  # to track arxiv_ids with missing domain values
    count_missing_domain_with_matches = 0  # to track arxiv_ids with missing domain values but matches in the epoch column
    count_incorrect_domain = 0  # to track arxiv_ids with incorrect domain values
    count_empty_domain = 0  # to track arxiv_ids with empty domain values
    
    correct_arxiv_ids = set()  # to track arxiv_ids with correct values 
    correct_na_arxiv_ids = set()  # to track arxiv_ids with correct n/a values 
    incorrect_arxiv_ids = set()  # to track arxiv_ids with incorrect values
    missing_data_epoch_arxiv_ids = set()  # to track arxiv_ids with missing data
    missing_data_extracted_arxiv_ids = set()  # to track arxiv_ids with missing data in extracted column
    missing_data_string_has_value_extracted_arxiv_ids = set()  # to track arxiv_ids where extracted is n/a but string is not
    incorrect_digits_arxiv_ids = set()  # to track arxiv_ids with incorrect digits
    incorrect_exponent_arxiv_ids = set()  # to track arxiv_ids with incorrect exponent

    incorrect_domain_arxiv_ids = set()  # to track arxiv_ids with incorrect domain values
    correct_domain_arxiv_ids = set()  # to track arxiv_ids with correct domain values
    missing_data_domain_no_matches_arxiv_ids = set()  # to track arxiv_ids with missing domain values but no matches in the epoch column
    missing_data_domain_with_matches_arxiv_ids = set()  # to track arxiv_ids
    empty_domain_arxiv_ids = set()  # to track arxiv_ids with empty domain values
    multimodal_only_arxiv_ids = set()  # to track arxiv_ids with only "Multimodal" in domain_overlap


    average_domain_jaccard_similarity = df['domain_jaccard_similarity'].mean() if 'domain_jaccard_similarity' in df.columns else None
    median_domain_jaccard_similarity = df['domain_jaccard_similarity'].median() if 'domain_jaccard_similarity' in df.columns else None
    max_domain_jaccard_similarity = df['domain_jaccard_similarity'].max() if 'domain_jaccard_similarity' in df.columns else None
    min_domain_jaccard_similarity = df['domain_jaccard_similarity'].min() if 'domain_jaccard_similarity' in df.columns else None
    count_non_one_or_zero_jaccard_similarities = df[(df['domain_jaccard_similarity'] != 1) & (df['domain_jaccard_similarity'] != 0)].shape[0] if 'domain_jaccard_similarity' in df.columns else None

    for arxiv_id in unique_arxiv_ids:
        rows = df[df['arxiv_id'] == arxiv_id]
        if not rows.empty:
           
            # i want to calculate for which arxiv_ids i have extracted the false values, i.e. the difference is not zero, but the extracted and epoch values are not n/a
            # sometimes an arxiv_id can have multiple rows (multiple models in the same paper), so we need to check all rows for the same arxiv_id
            # if one row has a zero difference, we count it as an arxiv_id with a zero difference -> correctly extracted values
            # if any rows has n/a in the difference column, we count it as an arxiv_id with missing data
            # if all rows have non-zero difference & are all differences are numeric (not n/a) -> incorrectly extracted values
            
            
            
            if (rows[diff_col] == 0).any():
                count_correct += 1
                correct_arxiv_ids.add(arxiv_id)
          
            elif (rows[parameters_first_digit_difference] == 0).any():
                count_incorrect_digits += 1
                incorrect_digits_arxiv_ids.add(arxiv_id)

            elif rows[diff_col].isna().any() and rows[extracted_col].notna().all():
                # count_arxiv_ids where the difference column is n/a but the extracted column is not
                count_missing_data_epoch += 1
                missing_data_epoch_arxiv_ids.add(arxiv_id)

            elif rows[diff_col].isna().any() and rows[epoch_col].notna().all(): 
                # count_arxiv_ids where the difference column is n/a but the epoch column is not

                #check if the 'parameters_string' column is not n/a
                if rows['parameters_string'].notna().all():
                    count_extracted_na_but_string_not_na += 1
                    missing_data_string_has_value_extracted_arxiv_ids.add(arxiv_id)
                else:
                    # count_arxiv_ids where the difference column is n/a and both extracted and epoch columns are n/a
                    count_missing_data_extracted += 1
                    missing_data_extracted_arxiv_ids.add(arxiv_id)
                
            elif (rows[diff_col] != 0).all():
                if rows[extracted_col].isna().all() and rows[epoch_col].isna().all():
                    # count_arxiv_ids where all rows have n/a in extracted and epoch columns
                    count_correct_na += 1
                    correct_na_arxiv_ids.add(arxiv_id)
                else:
                    if (abs(rows[parameters_factor_difference]) < 1).any():
                        # count_arxiv_ids where all rows have non-zero difference and are not n/a
                        count_incorrect_exponent += 1
                        incorrect_exponent_arxiv_ids.add(arxiv_id)
                    # count_arxiv_ids where all rows have non-zero difference and are not n/a
                    else:
                        count_incorrect += 1
                        incorrect_arxiv_ids.add(arxiv_id)

    # Count arxiv_ids with correct domain values
    #for arxiv_id in df['arxiv_id'].unique():
        #rows = df[df['arxiv_id'] == arxiv_id]

    num_of_entries = len(df)

    for i in range(len(df)):
        rows = df.iloc[[i]] 
        if not rows.empty:
            # Check if domain_overlap is not empty and domain_extracted_not_in_epoch, domain_epoch_not_in_extracted are empty
            if (rows['domain_overlap'].apply(lambda x: len(x) > 0).all() and
                rows['domain_extracted_not_in_epoch'].apply(lambda x: len(x) == 0).all() and
                rows['domain_epoch_not_in_extracted'].apply(lambda x: len(x) == 0).all()):
                count_correct_domain += 1
                correct_domain_arxiv_ids.add(arxiv_id)

            # Check if the domain_overlap is only "Multimodal" and one of the other two is empty (case if only "Multimodal" is present in one of the columns)
            elif ((rows['domain_overlap'].apply(lambda x: len(x) == 1 and 'Multimodal' in x).all() and
                  rows['domain_extracted_not_in_epoch'].apply(lambda x: len(x) == 0).all() and
                  rows['domain_epoch_not_in_extracted'].apply(lambda x: len(x) > 0).all()) or
                (rows['domain_overlap'].apply(lambda x: len(x) == 1 and 'Multimodal' in x).all() and
                    rows['domain_extracted_not_in_epoch'].apply(lambda x: len(x) > 0).all() and
                     rows['domain_epoch_not_in_extracted'].apply(lambda x: len(x) == 0).all())):
                count_multimodal_only += 1
                multimodal_only_arxiv_ids.add(arxiv_id)

            elif (rows['domain_overlap'].apply(lambda x: len(x) == 0).all() and
                  rows['domain_extracted_not_in_epoch'].apply(lambda x: len(x) == 0).all() and
                  rows['domain_epoch_not_in_extracted'].apply(lambda x: len(x) > 0).all()):
                count_missing_domain_no_matches += 1
                missing_data_domain_no_matches_arxiv_ids.add(arxiv_id)

            elif (rows['domain_overlap'].apply(lambda x: len(x) > 0).all() and
                  rows['domain_extracted_not_in_epoch'].apply(lambda x: len(x) == 0).all() and
                  rows['domain_epoch_not_in_extracted'].apply(lambda x: len(x) > 0).all()):
                count_missing_domain_with_matches += 1
                missing_data_domain_with_matches_arxiv_ids.add(arxiv_id)
            else:
                count_incorrect_domain += 1
                incorrect_domain_arxiv_ids.add(arxiv_id)

            # Check for empty domain values
            if (rows['domain_extracted'] == '').any() or (rows['domain_epoch'] == '').any():
                count_empty_domain += 1
                empty_domain_arxiv_ids.add(arxiv_id)

                
    
    '''
    print(f"  - Arxiv IDs with at least one zero difference: {count_with_zero}")
    print(f"  - Arxiv IDs with no zero differences: {count_without_zero}")
    print(f"  - Arxiv IDs with missing or NA values in '{diff_col}': {count_na_or_missing}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}': {df[extracted_col].isna().sum()}")
    print(f"  - Arxiv IDs with missing or NA values in '{epoch_col}': {df[epoch_col].isna().sum()}")
    print(f"  - Total missing or NA values across all relevant columns: {count_na_or_missing}")
    print(f"  - Total arxiv_ids checked: {len(zero_counts)}")
    print(f"  - Total arxiv_ids with missing or NA values in '{extracted_col}': {df[extracted_col].isna().sum()}")
    print(f"  - Total arxiv_ids with missing or NA values in '{epoch_col}': {df[epoch_col].isna().sum()}")
    print(f"  - Total arxiv_ids with missing or NA values in '{diff_col}': {df[diff_col].isna().sum()}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}' but not in '{epoch_col}': {count_extracted_na_only}")
    print(f"  - Arxiv IDs with missing or NA values in '{epoch_col}' but not in '{extracted_col}': {count_epoch_na_only}")
    print(f"  - Arxiv IDs with missing or NA values in both '{extracted_col}' and '{epoch_col}': {count_extracted_na_and_epoch_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}' and '{diff_col}': {count_extracted_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{epoch_col}' and '{diff_col}': {count_epoch_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in all three columns '{extracted_col}', '{epoch_col}' and '{diff_col}': {count_extracted_na_and_epoch_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}' and '{epoch_col}' but not in '{diff_col}': {count_extracted_na_and_epoch_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}' and '{diff_col}' but not in '{epoch_col}': {count_extracted_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{epoch_col}' and '{diff_col}' but not in '{extracted_col}': {count_epoch_na_and_diff_na}")
    print(f"  - Arxiv IDs with missing or NA values in '{extracted_col}', '{epoch_col}' and '{diff_col}': {count_extracted_na_and_epoch_na_and_diff_na}")
    print(f"  - Total arxiv_ids with missing or NA values in '{extracted_col}': {df[extracted_col].isna().sum()}")
    print(f"  - Total arxiv_ids with missing or NA values in '{epoch_col}': {df[epoch_col].isna().sum()}")
    print(f"  - Total arxiv_ids with missing or NA values in '{diff_col}': {df[diff_col].isna().sum()}")
    '''

    print(f"\nSummary:")
    #print(f"  - Number of unique arxiv_ids: {count_unique_arxiv_ids}")
    #print(f"Correctly extracted values (with zero differences): {count_with_zero}")
    #print(f"Correctly extracted values (incl. missing Values): {count_with_zero + count_extracted_na_and_epoch_na}")
    #print(f"Arxiv IDs with no zero differences: {count_without_zero}")
    
    print(f"Number of unique arxiv_ids: {count_unique_arxiv_ids}")
    print(5 * "=")
    print("--- Correct extracted ---")
    print(f"Correctly extracted values (row with with zero difference exists): {count_correct} - ({((count_correct) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(f"Arxiv IDs with correct n/a values: {count_correct_na} - ({((count_correct_na) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(f"Arxiv IDs with correct digits in middle of value: {count_incorrect_digits} - ({((count_incorrect_digits) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(2 * "-")
    print(f"Sum of all correctly extracted values: {count_correct + count_correct_na + count_incorrect_digits}")    
    print(f"Total Percentage: {((count_correct + count_correct_na + count_incorrect_digits) / count_unique_arxiv_ids) * 100:.2f}%")
    print(5 * "=")
    print("--- Incorrectly extracted ---")
    print(f"Incorrectly extracted values (all rows with non-zero differences): {count_incorrect} - ({((count_incorrect) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(f"Arxiv IDs with incorrect exponent values: {count_incorrect_exponent} - ({((count_incorrect_exponent) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(f"Arxiv IDs with missing data in epoch column: {count_missing_data_epoch} - ({((count_missing_data_epoch) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(f"Arxiv IDs with missing data in extracted column: {count_missing_data_extracted} - ({((count_missing_data_extracted) / count_unique_arxiv_ids) * 100:.2f}%)") 
    print(2 * "")
    print(f"Sum of all incorrectly extracted values: {count_incorrect + count_missing_data_epoch + count_missing_data_extracted + count_incorrect_exponent}")
    print(f"Total Percentage: {((count_incorrect + count_missing_data_epoch + count_missing_data_extracted + count_incorrect_exponent) / count_unique_arxiv_ids) * 100:.2f}%")
    print(5 * "")
    print("--- Unclear data ---")
    print(f"Arxiv IDs with extracted column n/a but string column has value: {count_extracted_na_but_string_not_na} - ({((count_extracted_na_but_string_not_na) / count_unique_arxiv_ids) * 100:.2f}%)")
    print(2* "-")
    print(f"Total arxiv_ids with unclear data: {count_extracted_na_but_string_not_na}")
    print(f"Total Percentage of unclear data: {((count_extracted_na_but_string_not_na) / count_unique_arxiv_ids) * 100:.2f}%")
    print(5 * "=")
    print("--- Domain values ---")
    print(f"Arxiv IDs with correct domain values: {count_correct_domain} - ({((count_correct_domain) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with only 'Multimodal' in domain_overlap: {count_multimodal_only} - ({((count_multimodal_only) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with additional domain values: {count_additional_domain} - ({((count_additional_domain) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with missing domain values (no matches): {count_missing_domain_no_matches} - ({((count_missing_domain_no_matches) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with missing domain values (with matches): {count_missing_domain_with_matches} - ({((count_missing_domain_with_matches) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with incorrect domain values: {count_incorrect_domain} - ({((count_incorrect_domain) / num_of_entries) * 100:.2f}%)")
    print(f"Arxiv IDs with empty domain values: {count_empty_domain} - ({((count_empty_domain) / num_of_entries) * 100:.2f}%)")
    print("--- Domain Jaccard Similarity ---")
    print(f"Average Jaccard similarity of domain values: {average_domain_jaccard_similarity:.2f}")
    print(f"Median Jaccard similarity of domain values: {median_domain_jaccard_similarity:.2f}")
    print(f"Max Jaccard similarity of domain values: {max_domain_jaccard_similarity:.2f}")
    print(f"Min Jaccard similarity of domain values: {min_domain_jaccard_similarity:.2f}")
    print(f"Number of Jaccard similarities that are neither 0 nor 1: {count_non_one_or_zero_jaccard_similarities}")
    print(5 * "=")


    #print(f"Incorrectly extracted values (all rows with non-zero differences): {count_incorrect}")  
    #print(f"Arxiv IDs with no data in epoch column: {count_missing_data_epoch}")
    #print(f"Arxiv IDs with extracted column n/a but string column has value: {count_extracted_na_but_string_not_na}")
    #print(f"Arxiv IDs with no data in extracted column: {count_missing_data_extracted}")
    
    


    #print(f"Total arxiv_ids checked: {len(zero_counts)}")

    return correct_arxiv_ids, correct_na_arxiv_ids, incorrect_arxiv_ids, missing_data_epoch_arxiv_ids, missing_data_extracted_arxiv_ids, missing_data_string_has_value_extracted_arxiv_ids, incorrect_digits_arxiv_ids, incorrect_domain_arxiv_ids, missing_data_domain_no_matches_arxiv_ids, missing_data_domain_with_matches_arxiv_ids

correct_arxiv_ids, correct_na_arxiv_ids, incorrect_arxiv_ids, missing_data_epoch_arxiv_ids, missing_data_extracted_arxiv_ids, missing_data_string_has_value_extracted_arxiv_ids, incorrect_digits_arxiv_ids, incorrect_domain_arxiv_ids, missing_data_domain_no_matches_arxiv_ids, missing_data_domain_with_matches_arxiv_ids = check_zero_differences(comparison_table, prefix='parameters')



Summary:
Number of unique arxiv_ids: 432
=====
--- Correct extracted ---
Correctly extracted values (row with with zero difference exists): 167 - (38.66%)
Arxiv IDs with correct n/a values: 79 - (18.29%)
Arxiv IDs with correct digits in middle of value: 26 - (6.02%)
--
Sum of all correctly extracted values: 272
Total Percentage: 62.96%
=====
--- Incorrectly extracted ---
Incorrectly extracted values (all rows with non-zero differences): 45 - (10.42%)
Arxiv IDs with incorrect exponent values: 22 - (5.09%)
Arxiv IDs with missing data in epoch column: 9 - (2.08%)
Arxiv IDs with missing data in extracted column: 12 - (2.78%)

Sum of all incorrectly extracted values: 88
Total Percentage: 20.37%

--- Unclear data ---
Arxiv IDs with extracted column n/a but string column has value: 72 - (16.67%)
--
Total arxiv_ids with unclear data: 72
Total Percentage of unclear data: 16.67%
=====
--- Domain values ---
Arxiv IDs with correct domain values: 945 - (78.29%)
Arxiv IDs with only 'Multimodal' in 

In [ ]:
#create dataframe with only incorrectly extracted values, i.e. where the difference is not zero, but the extracted and epoch values are not n/a
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
    

    


if incorrect_arxiv_ids:
    incorrect_df = comparison_table[comparison_table['arxiv_id'].isin(incorrect_arxiv_ids)][['arxiv_id', 'model_name_extracted', 'model_name_epoch', 'parameters_extracted', 'parameters_epoch', 'parameters_difference', 'parameters_percentage_difference', 'parameters_first_digit_difference', 'parameters_string', 'parameters_factor_difference']]
    print(f"\nIncorrectly extracted values DataFrame shape: {incorrect_df.shape}")
    
    if not incorrect_df.empty:
        print(f"\nSample incorrectly extracted values:")
        #display(incorrect_df)
    else:
        print("No incorrectly extracted values found!")

if missing_data_epoch_arxiv_ids:
    missing_data_df = comparison_table[comparison_table['arxiv_id'].isin(missing_data_epoch_arxiv_ids)][['arxiv_id', 'model_name_extracted', 'model_name_epoch', 'parameters_extracted', 'parameters_epoch', 'parameters_difference', 'parameters_percentage_difference', 'parameters_first_digit_difference']]
    print(f"\nMissing data DataFrame shape: {missing_data_df.shape}")
    
    if not missing_data_df.empty:
        print(f"\nSample missing data without epoch params:")
        #display(missing_data_df.head(10))
    else:
        print("No missing data found!")

if missing_data_extracted_arxiv_ids:
    params_as_string_df = comparison_table[comparison_table['arxiv_id'].isin(missing_data_extracted_arxiv_ids)][['arxiv_id', 'base_model_extracted', 'model_name_extracted', 'model_name_epoch', 'parameters_string', 'parameters_extracted', 'parameters_epoch', 'parameters_difference', 'parameters_percentage_difference', 'parameters_first_digit_difference']]
    params_as_string_df = params_as_string_df[params_as_string_df['parameters_extracted'].isna() 
                                                          & params_as_string_df['parameters_string'].notna() 
                                                          & (params_as_string_df['parameters_string'].astype(str).str.strip().str.lower() != 'n/a')]
    print(f"\nMissing data DataFrame shape (extracted): {params_as_string_df.shape}")
    
    if not params_as_string_df.empty:
        print(f"\nSample missing data without extracted params:")
        #display(params_as_string_df.head(5))
    else:
        print("No missing data found in extracted params!")

if not params_as_string_df.empty:
    params_as_string_where_epoch_not_na = params_as_string_df[params_as_string_df['parameters_epoch'].notna()]
    params_as_string_where_epoch_na = params_as_string_df[params_as_string_df['parameters_epoch'].isna()]

    print(f"\nParameters as string where epoch is not NA: {params_as_string_where_epoch_not_na.shape}")
    if not params_as_string_where_epoch_not_na.empty:
        print(f"\nSample parameters as string where epoch is not NA:")
        #display(params_as_string_where_epoch_not_na)
    else:
        print("No parameters as string where epoch is not NA found!")

    print(f"\nParameters as string where epoch is NA: {params_as_string_where_epoch_na.shape}")
    if not params_as_string_where_epoch_na.empty:
        print(f"\nSample parameters as string where epoch is NA:")
        #display(params_as_string_where_epoch_na)

    else:
        print("No parameters as string where epoch is NA found!")

if missing_data_string_has_value_extracted_arxiv_ids:
    missing_data_string_df = comparison_table[comparison_table['arxiv_id'].isin(missing_data_string_has_value_extracted_arxiv_ids)][['arxiv_id', 'base_model_extracted', 'model_name_extracted', 'model_name_epoch', 'parameters_string', 'parameters_extracted', 'parameters_epoch', 'parameters_difference', 'parameters_percentage_difference', 'parameters_first_digit_difference']]
   
    print(f"\nMissing data DataFrame shape (string has value): {missing_data_string_df.shape}")
    if not missing_data_string_df.empty:
        print(f"\nSample missing data where extracted is n/a but string has value:")
        #display(missing_data_string_df)
    else:
        print("No missing data found where extracted is n/a but string has value!")

if incorrect_domain_arxiv_ids:
    incorrect_domain_df = comparison_table[comparison_table['arxiv_id'].isin(incorrect_domain_arxiv_ids)][['arxiv_id', 'base_model_extracted', 'model_name_extracted', 'model_name_epoch', 'domain_extracted', 'domain_epoch', 'domain_overlap', 'domain_extracted_not_in_epoch', 'domain_epoch_not_in_extracted', 'domain_jaccard_similarity']]
    print(f"\nIncorrect domain DataFrame shape: {incorrect_domain_df.shape}")
    if not incorrect_domain_df.empty:
        print(f"\nSample incorrect domain data:")
        display(incorrect_domain_df)
    else:
        print("No incorrect domain data found!")

    # Reset pandas display options
    pd.reset_option('display.max_columns')
    pd.reset_option('display.max_rows')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')


In [17]:
#print col names in comparison_table
print(f"\nColumn names in comparison_table:")
for i, col in enumerate(comparison_table.columns):
    print(f"{i+1:2d}. {col}")


Column names in comparison_table:


In [18]:
def calculate_na_values_by_run_range(run_id_start, run_id_end, na_values=None):
    """
    Calculate the number of N/A values for each extracted field within a given range of extraction runs.
    
    Args:
        run_id_start (int): Starting extraction run ID (inclusive)
        run_id_end (int): Ending extraction run ID (inclusive)
        na_values (list, optional): List of values to consider as N/A. 
                                  Default: ['n/a', '', 'N/A', 'unknown', 'None', 'null', 
                                           'not specified', 'not available', 'not reported', 
                                           'unclear', '-', 'not mentioned', 'not found']
    
    Returns:
        pandas.DataFrame: DataFrame with columns ['field_name', 'total_extractions', 'na_count', 'na_percentage', 'valid_count']
    """
    import pandas as pd
    from db.db_utils import get_connection
    
    # Default N/A values if not specified
    if na_values is None:
        na_values = ['n/a', '', 'N/A', 'unknown', 'None', 'null', 
                     'not specified', 'not available', 'not reported', 
                     'unclear', '-', 'not mentioned', 'not found', '{n/a}']
    
    print(f" Analyzing N/A values for extraction runs {run_id_start} to {run_id_end}")
    print(f" N/A values considered: {na_values}")
    
    conn = None
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        # Query to get all extracted fields within the run range
        query = """
        SELECT 
            ef.field_name,
            ef.value,
            ef.run_id
        FROM extracted_fields ef
        JOIN extraction_runs er ON ef.run_id = er.id
        WHERE ef.run_id >= %s AND ef.run_id <= %s
        ORDER BY ef.field_name, ef.run_id
        """
        
        cursor.execute(query, (run_id_start, run_id_end))
        results = cursor.fetchall()
        
        if not results:
            print(f" No extraction results found for run range {run_id_start}-{run_id_end}")
            return pd.DataFrame()
        
        # Convert to DataFrame for easier analysis
        df = pd.DataFrame(results, columns=['field_name', 'value', 'run_id'])
        print(f" Found {len(df)} extracted fields across {len(df['run_id'].unique())} runs")
        
        # Group by field_name and calculate N/A statistics
        field_stats = []
        
        for field_name in df['field_name'].unique():
            field_data = df[df['field_name'] == field_name]
            total_extractions = len(field_data)
            
            # Count N/A values (including None/NULL values)
            na_mask = (
                field_data['value'].isin(na_values) | 
                field_data['value'].isna() |
                field_data['value'].isnull()
            )
            na_count = na_mask.sum()
            valid_count = total_extractions - na_count
            na_percentage = (na_count / total_extractions * 100) if total_extractions > 0 else 0
            
            field_stats.append({
                'field_name': field_name,
                'total_extractions': total_extractions,
                'na_count': na_count,
                'na_percentage': round(na_percentage, 2),
                'valid_count': valid_count
            })
        
        # Create results DataFrame
        results_df = pd.DataFrame(field_stats)
        results_df = results_df.sort_values('na_percentage', ascending=False)
        
        # Display summary
        print(f"\n N/A Analysis Summary for runs {run_id_start}-{run_id_end}:")
        print("=" * 80)
        print(f"{'Field Name':<25} {'Total':<8} {'N/A':<6} {'N/A %':<8} {'Valid':<6}")
        print("-" * 80)
        
        for _, row in results_df.iterrows():
            print(f"{row['field_name']:<25} {row['total_extractions']:<8} {row['na_count']:<6} {row['na_percentage']:<8}% {row['valid_count']:<6}")
        
        # Overall statistics
        total_all_extractions = results_df['total_extractions'].sum()
        total_na_count = results_df['na_count'].sum()
        overall_na_percentage = (total_na_count / total_all_extractions * 100) if total_all_extractions > 0 else 0
        
        print("-" * 80)
        print(f"{'OVERALL':<25} {total_all_extractions:<8} {total_na_count:<6} {overall_na_percentage:<8.2f}% {total_all_extractions - total_na_count:<6}")
        
        # Highlight fields with high N/A rates
        high_na_fields = results_df[results_df['na_percentage'] > 50]
        if not high_na_fields.empty:
            print(f"\n️  Fields with >50% N/A values ({len(high_na_fields)} fields):")
            for _, row in high_na_fields.iterrows():
                print(f"   • {row['field_name']}: {row['na_percentage']}% N/A")
        
        # Highlight fields with low N/A rates
        low_na_fields = results_df[results_df['na_percentage'] < 10]
        if not low_na_fields.empty:
            print(f"\n Fields with <10% N/A values ({len(low_na_fields)} fields):")
            for _, row in low_na_fields.iterrows():
                print(f"   • {row['field_name']}: {row['na_percentage']}% N/A")
        
        return results_df
        
    except Exception as e:
        print(f" Error analyzing N/A values: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

print(" N/A analysis function loaded!")

 N/A analysis function loaded!


In [19]:
# Example usage of the N/A analysis function
print(" Example: Analyzing N/A values for extraction runs")

# Example 1: Analyze a specific range of runs
# Replace these with actual run IDs from your database
example_start_run = 314
example_end_run = 9045

print(f"\n Analyzing runs {example_start_run} to {example_end_run}...")

# Uncomment the line below to run the analysis
na_analysis_results = calculate_na_values_by_run_range(example_start_run, example_end_run)

print("\n Usage Tips:")
print("1. Use get_run_id_ranges() first to see available run ID ranges")
print("2. Focus on specific extraction batches by adjusting start/end run IDs")
print("3. Customize na_values parameter to include your specific N/A indicators")
print("4. Results are sorted by N/A percentage (highest first)")
print("5. Use the returned DataFrame for further analysis or visualization")

print("\n Example with custom N/A values:")
print("# custom_na_values = ['n/a', 'unknown', 'not found', 'missing', '']")
print("# results = calculate_na_values_by_run_range(100, 200, na_values=custom_na_values)")

 Example: Analyzing N/A values for extraction runs

 Analyzing runs 314 to 9045...
 Analyzing N/A values for extraction runs 314 to 9045
 N/A values considered: ['n/a', '', 'N/A', 'unknown', 'None', 'null', 'not specified', 'not available', 'not reported', 'unclear', '-', 'not mentioned', 'not found', '{n/a}']
 Found 508376 extracted fields across 8703 runs

 N/A Analysis Summary for runs 314-9045:
Field Name                Total    N/A    N/A %    Valid 
--------------------------------------------------------------------------------
hardware_utilization      25532    25079  98.23   % 453   
publication_date          25568    23266  91.0    % 2302  
training_time             25658    21150  82.43   % 4508  
epochs                    25748    19067  74.05   % 6681  
hardware_quantity         25779    16686  64.73   % 9093  
base_model                25772    14976  58.11   % 10796 
training_dataset_size     25944    13950  53.77   % 11994 
training_hardware         25880    13628  52.6